## 2. Download Proxies Workflow

1. Packages
2. Comments
3. Settings
4. Area of Interest & Tiles
5. Compute Satellite Derived Bathymetry

### 1. Packages

In [1]:
# Generic packages
import folium
import geopandas as gpd
import numpy as np
import os
import sys
import time
import pickle
from tqdm import tqdm

# GEE specific packages
project = "cmems-sdb-11209821-002" #'bathymetry'
import ee
try:
    ee.Initialize(project=project)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=project)

# custom functionality import without requirement to pip install package
dir_path_ee_packages = os.path.join(os.path.expanduser('~'), 'Documents', 'GitHub', 'ee-packages-py') # path to local GitHub clone
sys.path.append(dir_path_ee_packages)
from eepackages.applications.bathymetry import Bathymetry
from eepackages import tiler

C:\Users\kras\AppData\Local\Temp\ipykernel_16476\1406294515.py:3: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### 2. Comments

Acknowledgements & code references:
- https://github.com/openearth/eo-bathymetry/
- https://github.com/openearth/eo-bathymetry-functions/
- https://github.com/gee-community/ee-packages-py

In [2]:
# TODO list
# TODO: look if scale / crs does not influence the output used before exporting as we have differences between the GEE export and the local post-processed export

### 3. Settings

In [3]:
# Settings
run_mode = 'global'                      # Run mode, either 'local' or 'global'
project_name = 'AOI_WestEurope_v2'      # Name of the project AoI, or one in the folder
mode = 'intertidal_improved_100m_global'  # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
dir_path_output = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', f'{mode}')                                                         # Output directory
file_path_aoi = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_upscale', '{}.geojson'.format(project_name.replace('_v2','')))                           # AOI file
file_path_mask = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result.parquet')                     # Mask file
file_path_mask_ed = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result_erosion_dilation.parquet') # Mask (erosion/dilation) file
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered_v2.parquet')                  # Tiles file
file_path_credentials = os.path.join(dir_path_base, '00_miscellaneous', 'KEYS', "cmems-sdb-11209821-002-d08744ac2a69.json") #'bathymetry-543b622ddce7.json'   # Cloud Storage credentials file
file_path_progress = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', 'progress_{}'.format(run_mode))                                 # progress dir

# Google Cloud Bucket
bucket = "cmems-isdb" #'cmems-sdb'

# Load Google credentials
if not file_path_credentials == '':  
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = file_path_credentials

# load GTSM & gebco data
#gtsm_col = ee.FeatureCollection('projects/bathymetry/assets/gtsm_waterlevels_2021_v2') # Loaded in bathymetry
#gebco_image = ee.Image('projects/bathymetry/assets/gebco_2023_hat_lat') # Loaded in bathymetry

### 4. Area of Interest & Tiles

In [283]:
# Read geometries
gdf_aoi = gpd.read_file(file_path_aoi)
gdf_mask = gpd.read_parquet(file_path_mask)
gdf_mask_ed = gpd.read_parquet(file_path_mask_ed)
gdf_tiles = gpd.read_parquet(file_path_tiles)

In [284]:
project_name = "NWN" 

if run_mode == 'local':   

    # Get mask where pixel value is 3.0
    gdf_mask = gdf_mask[gdf_mask['pixel_value'] == 3.0]
    gdf_mask_ed = gdf_mask_ed[gdf_mask_ed['pixel_value'] == 3.0]

    # Get mask that intersects with the area of interest
    gdf_mask = gdf_mask[gdf_mask.intersects(gdf_aoi.unary_union)]
    gdf_mask_ed = gdf_mask_ed[gdf_mask_ed.intersects(gdf_aoi.unary_union)]

    # Clip mask to area of interest
    gdf_mask = gpd.overlay(gdf_mask, gdf_aoi, how='intersection')
    gdf_mask_ed = gpd.overlay(gdf_mask_ed, gdf_aoi, how='intersection')

    # Get tiles that intersect with mask
    gdf_tiles = gdf_tiles[gdf_tiles.intersects(gdf_mask_ed.unary_union)]

    # Select tiles within bounds (France)
    #bounds = [-5.0, 45.5, 0, 47.5]
    #gdf_tiles = gdf_tiles.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

    # Select specific tile
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x507_y363']
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x528_y331']

    # Sort tiles based on intertidal coverage
    gdf_tiles = gdf_tiles.sort_values(by='intertidal_coverage_ed', ascending=False)

    # Print tiles
    print('Number of tiles: {}'.format(len(gdf_tiles)))
    gdf_tiles.head(5)

if run_mode == 'global':

    # Get mask where pixel value is 3.0
    #gdf_mask = gdf_mask[gdf_mask['pixel_value'] == 3.0]
    #gdf_mask_ed = gdf_mask_ed[gdf_mask_ed['pixel_value'] == 3.0]

    # Get mask that intersects with the area of interest
    #gdf_mask = gdf_mask[gdf_mask.intersects(gdf_aoi.unary_union)]
    #gdf_mask_ed = gdf_mask_ed[gdf_mask_ed.intersects(gdf_aoi.unary_union)]

    # Clip mask to area of interest
    #gdf_mask = gpd.overlay(gdf_mask, gdf_aoi, how='intersection')
    #gdf_mask_ed = gpd.overlay(gdf_mask_ed, gdf_aoi, how='intersection')

    # Get tiles that intersect with mask
    #gdf_tiles = gdf_tiles[gdf_tiles.intersects(gdf_mask_ed.unary_union)]

    # Select tiles within bounds (France)
    #bounds = [-5.0, 45.5, 0, 47.5]
    #gdf_tiles = gdf_tiles.cx[bounds[0]:bounds[2], bounds[1]:bounds[3]]

    # Select specific tile
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x507_y363']
    #gdf_tiles = gdf_tiles[gdf_tiles['name'] == 'z10_x528_y331']

    # filter the GDF on specific criteria related to the intertidal coverage & distance to a GTSM station
    gdf_tiles_red = gdf_tiles[gdf_tiles["intertidal_coverage_ed"]*100 > 0] # percentages
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["intertidal_coverage"]*100 >= 1] # percentages
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["nearest_station_distance"] <= 37000] # m, 37000 is at Z10 at most on the corner-point of the adjacent tile from the centroid
    
    # count number of occurences ids in reg_regions
    #print(gdf_tiles_red['ref_region'].value_counts())

    # select specific area
    gdf_tiles_red = gdf_tiles_red[gdf_tiles_red["ref_region"] == project_name]

    # Sort tiles based on intertidal coverage
    gdf_tiles_red = gdf_tiles_red.sort_values(by='intertidal_coverage_ed', ascending=False)

    # Print tiles
    print('Number of tiles: {}'.format(len(gdf_tiles_red)))
    gdf_tiles_red.head(5)

    # put to gdf_tiles
    gdf_tiles = gdf_tiles_red

Number of tiles: 1518


In [289]:
gdf_tiles = gdf_tiles[1400:]

In [290]:
gdf_tiles

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed
23997,z10_x199_y240,12630,199.0,240.0,10,"POLYGON ((-110.03906 68.52823, -109.68750 68.5...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,2.17,0.1,station 07080,-109.084200,68.721990,34665.462703,0.037538,0.000986
24185,z10_x110_y297,5923,110.0,297.0,10,"POLYGON ((-141.32812 59.88894, -140.97656 59.8...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.61,0.1,station 11457,-141.455000,59.893390,19247.328419,0.020685,0.000983
23796,z10_x179_y200,13725,179.0,200.0,10,"POLYGON ((-117.07031 73.12495, -116.71875 73.1...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.78,0.1,station 07094,-116.022100,73.139440,28397.005522,0.040535,0.000980
23393,z10_x129_y313,4668,129.0,313.0,10,"POLYGON ((-134.64844 56.94497, -134.29687 56.9...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.59,0.1,station 11432,-134.411000,57.067580,4785.158165,0.047418,0.000976
24949,z10_x190_y202,14882,190.0,202.0,10,"POLYGON ((-113.20312 72.91964, -112.85156 72.9...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,2.90,0.1,station 07096,-112.844600,73.000620,6791.002086,0.046133,0.000970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23999,z10_x130_y316,4720,130.0,316.0,10,"POLYGON ((-134.29687 56.36525, -133.94531 56.3...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.40,0.0,station 11423,-134.161600,56.502260,5085.637495,0.110460,0.000067
22819,z10_x128_y307,4613,128.0,307.0,10,"POLYGON ((-135.00000 58.07788, -134.64844 58.0...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.20,0.0,station 21824,-134.311523,58.256835,31524.023932,0.050208,0.000049
24075,z10_x201_y158,15993,201.0,158.0,10,"POLYGON ((-109.33594 76.92061, -108.98437 76.9...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.95,0.0,station 20887,-109.042969,76.889642,8388.953983,0.043432,0.000024
24408,z10_x60_y318,1292,60.0,318.0,10,"POLYGON ((-158.90625 55.97380, -158.55469 55.9...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,NWN,0.35,0.0,station 11493,-158.761300,55.929650,15934.393108,0.125621,0.000023


In [275]:
# Plot area of interest
# m = folium.Map(location=[gdf_aoi.centroid.y, gdf_aoi.centroid.x], zoom_start=6)
# m = gdf_aoi.explore(m=m, style_kwds={'color': 'red', 'fillOpacity': 0.2}, name='Area of Interest', tooltip=False)
# m = gdf_mask.explore(m=m, style_kwds={'color': 'blue', 'fillOpacity': 0.2}, name='Mask', tooltip=False)
# m = gdf_mask_ed.explore(m=m, style_kwds={'color': 'purple', 'fillOpacity': 0.2}, name='Mask Erosion Dilation', tooltip=False)
# m = gdf_tiles.explore(m=m, cmap='Greens', column='intertidal_coverage_ed', name='Tiles', vmin=0, vmax=np.percentile(gdf_tiles['intertidal_coverage_ed'], 98), tooltip=['id', 'name', 'intertidal_coverage_ed'], 
#                          legend=True)
# folium.LayerControl().add_to(m)
#m

### 5. Compute Satellite Derived Bathymetry

In [291]:
# functions to compute sub & intertidal bathymetry proxies based on standardized SlippyMap tiling practice
# functions taken from: https://github.com/openearth/eo-bathymetry/blob/master/notebooks/rws-bathymetry/export_bathymetry.ipynb
# resembles similar behaviour as in https://github.com/openearth/eo-bathymetry-functions but slightly adjusted for local study 

# Packages
from typing import Optional, List, Dict, Any
from logging import Logger, getLogger
from googleapiclient.discovery import build
from re import sub
from ctypes import ArgumentError
from functools import partial
from dateutil.parser import parse

logger: Logger = getLogger(__name__)

def get_tile_intertidal_bathymetry(tile: ee.Feature, start: ee.String, stop: ee.String) -> ee.Image:
    """
    Get intertidal bathymetry based on tile geometry.
    Server-side compliant for GEE.

    args:
        tile (ee.Feature): tile geometry used to obtain bathymetry.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
    
    returns:
        ee.Image: image containing intertidal bathymetry covering tile.
    """

    bounds: ee.Geometry = ee.Feature(tile).geometry().bounds(1)
    sdb: Bathymetry = Bathymetry()
    zoom: ee.String = ee.String(tile.get("zoom"))
    tx: ee.String = ee.String(tile.get("tx"))
    ty: ee.String = ee.String(tile.get("ty"))
    tile_name: ee.String = ee.String("z").cat(zoom).cat("_x").cat(tx).cat("_y").cat(ty).replace("\.\d+", "", "g")
    img_fullname: ee.String = ee.String(tile_name).cat("_t").cat(ee.Date(start).millis().format())
        
    image: ee.Image = sdb.compute_intertidal_depth(
        bounds=bounds,
        start=start,
        stop=stop,
        scale=tiler.zoom_to_scale(ee.Number.parse(tile.get("zoom"))).multiply(5), # scale to search for clean images
        # missions=['S2', 'L8'],
        # filter: ee.Filter.dayOfYear(7*30, 9*30), # summer-only
        filter_masked=False, 
        tile=tile,
        # filterMaskedFraction = 0.5,
        # skip_scene_boundary_fix=False,
        # skip_neighborhood_search=False,
        neighborhood_search_parameters={"erosion": 0, "dilation": 0, "weight": 50},
        bounds_buffer=0,
        water_index_min=-0.05,
        water_index_max=0.15,
        # lowerCdfBoundary=45,
        # upperCdfBoundary=50,
        # cloud_frequency_threshold_data=0.15, 
        clip = True,
        mosaic_by_day = True
    )# .reproject(ee.Projection("EPSG:3857").atScale(90))

    image = image.set(
        "fullname", img_fullname,
        "system:time_start", ee.Date(start).millis(),
        "system:time_stop", ee.Date(stop).millis(),
        "zoom", zoom,
        "tx", tx,
        "ty", ty
    )

    return image

def tile_to_asset(
    image: ee.Image,
    tile: ee.Feature,
    export_scale: int,
    asset_path_prefix: str,
    asset_name: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    
    asset_id: str = f"{asset_path_prefix}/{asset_name}"
    asset: Dict[str, Any] = ee.data.getInfo(asset_id)
    if overwrite and asset:
        logger.info(f"deleting asset {asset}")
        ee.data.deleteAsset(asset_id)
    elif asset:
        logger.info(f"asset {asset} already exists, skipping {asset_name}")
        return
    task: ee.batch.Task = ee.batch.Export.image.toAsset(
        image,
        assetId=asset_id,
        description=asset_name,
        region=tile.geometry(),
        scale=export_scale,
        maxPixels= 1e10
    )
    task.start()
    logger.info(f"exporting {asset_name} to {asset_id}")

def tile_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    crs: str,
    export_scale: int,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return
        
    task: ee.batch.Task = ee.batch.Export.image.toCloudStorage(
        image,
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        region=tile.geometry(),
        scale=export_scale,
        crs=crs,
        fileFormat='GeoTIFF',
        formatOptions= {'cloudOptimized': True}, # enables easy QGIS plotting
        maxPixels= 1e10
    )
    task.start()
    return task

def metadata_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return
    
    meta_feature = ee.Feature(None, image.toDictionary().set("tx", tile.get("tx")).set("ty", tile.get("ty")))

    task: ee.batch.Task = ee.batch.Export.table.toCloudStorage(
        ee.FeatureCollection(meta_feature),
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        fileFormat='csv',
        maxVertices=0
    )
    task.start()
    return task

def export_sdb_tiles(
    sink: str,
    tile_list: ee.List,
    num_tiles: int,
    export_scale: int,
    crs: str,
    sdb_tiles: ee.ImageCollection,
    name_suffix: str,
    mode: str,
    task_list: List[ee.batch.Task],
    overwrite: bool,
    bucket: Optional[str] = None
) -> List[ee.batch.Task]:
    """
    Export list of tiled images containing sub or intertidal tidal bathymetry. Fires off the tasks and adds to the list of tasks.
    based on: https://github.com/gee-community/gee_tools/blob/master/geetools/batch/imagecollection.py#L166

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        tile_list (ee.List): list of tile features.
        num_tiles (int): number of tiles in `tile_list`.
        scale (int): scale of the export product.
        sdb_tiles (ee.ImageCollection): collection of subtidal bathymetry images corresponding
            to input tiles.
        name_suffix (str): unique identifier after tile statistics.
        task_list (List[ee.batch.Task]): list of tasks, adds tasks created to this list.
        overwrite (bool): whether to overwrite the current assets under the same `asset_path`.
        bucket (str): Bucket where the data is stored. Only used when sink = "cloud"
    
    returns:
        List[ee.batch.Task]: list of started tasks

    """
    if sink == "asset":
        user_name: str = ee.data.getAssetRoots()[0]["id"].split("/")[-1]
        asset_path_prefix: str = f"users/{user_name}/eo-bathymetry"
        ee.data.create_assets(asset_ids=[asset_path_prefix], asset_type="Folder", mk_parents=True)
    
    for i in range(num_tiles):
        # get tile
        temp_tile: ee.Feature = ee.Feature(tile_list.get(i))
        tile_metadata: Dict[str, Any] = temp_tile.getInfo()["properties"]
        tx: str = tile_metadata["tx"]
        ty: str = tile_metadata["ty"]
        zoom: str = tile_metadata["zoom"]
        # filter imagecollection based on tile
        filtered_ic: ee.ImageCollection = sdb_tiles \
            .filterMetadata("tx", "equals", tx) \
            .filterMetadata("ty", "equals", ty) \
            .filterMetadata("zoom", "equals", zoom)
        # if filtered correctly, only a single image remains
        img: ee.Image = ee.Image(filtered_ic.first())  # have to cast here
        img_name: str = sub(r"\.\d+", "", f"{mode}/z{zoom}/x{tx}/y{ty}/") + name_suffix 
        print("Submitting task for tile: ", img_name)
        # Export images
        if sink == "asset":  # Replace with case / switch in python 3.10
            task_img: Optional[ee.batch.Task] = tile_to_asset(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                asset_path_prefix=asset_path_prefix,
                asset_name=img_name.replace("/","_"),
                overwrite=overwrite
            )
            if task_img: task_list.append(task_img)
        elif sink == "cloud":
            if not bucket:
                raise ArgumentError("Sink option requires \"bucket\" arg.")
            task_img: ee.batch.Task = tile_to_cloud_storage(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                crs=crs, 
                bucket=bucket,
                bucket_path=img_name,
                overwrite=overwrite
            )

            task_meta: ee.batch.Task = metadata_to_cloud_storage(
                image=img,
                tile=temp_tile,
                bucket=bucket,
                bucket_path=sub(r"\.\d+", "", f"{mode}_meta/z{zoom}/x{tx}/y{ty}/") + name_suffix,
                overwrite=overwrite
            )
        else:
            raise ArgumentError("unrecognized data sink: {sink}")
        task_list.append(task_img)
        task_list.append(task_meta)
    return task_list

def export_tiles(
    sink: str,
    mode: str,
    geometry: ee.Geometry,
    zoom: int,
    start: str,
    stop: str,
    scale: Optional[float] = None,
    crs: str = "EPSG:4326",
    buf_pix: int = 0,
    step_months: int = 3,
    window_months: int = 24,
    overwrite: bool = False,
    bucket: Optional[str] = None
) -> None:
    """
    From a geometry, creates tiles of input zoom level, calculates subtidal bathymetry in those
    tiles, and exports those tiles.

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        mode (str): either "subtidal" or "intertidal" for select type of bathymetry to export.
        geometry (ee.Geometry): geometry of the area of interest.
        zoom (int): zoom level of the to-be-exported tiles.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
        scale Optional(float): scale of the product to be exported. Defaults tiler.zoom_to_scale(zoom).getInfo().
        crs (str): projection of the output image.
        buf_pix (int): buffer around the tile (in pixels).
        step_months (int): steps with which to roll the window over which the subtidal bathymetry
            is calculated.
        windows_months (int): number of months over which the bathymetry is calculated.
    """

    # Function to create a window
    def create_year_window(year: ee.Number, month: ee.Number) -> ee.Dictionary:
        t: ee.Date = ee.Date.fromYMD(year, month, 1)
        d_format: str = "YYYY-MM-dd"
        return ee.Dictionary({
            "start": t.format(d_format),
            "stop": t.advance(window_months, 'month').format(d_format)
            })
    
    window_length: int = (parse(stop).year-parse(start).year)*12+(parse(stop).month-parse(start).month) # in months
    dates: ee.List = ee.List.sequence(parse(start).year, parse(stop).year-window_months/12).map(
        lambda year: ee.List.sequence(1, None, step_months, int((window_length-window_months)/step_months)+1).map(partial(create_year_window, year))
    ).flatten() # NOTE, still buggy, works for yearly composites. Not nice for end_date "2022-03-01"; error Date.fromYMD: Bad year/month/day: 2021/13/1.

    dates = ee.List([dates.get(0)]) #ADJUSTED TO SELECT FIRST DATE ONLY
    
    # Get tiles
    tile: ee.Feature =  ee.Feature(geometry.buffer(buf_pix*scale/111120, ee.ErrorMargin((buf_pix*scale*0.01)/111120, 'projected'), proj="EPSG:4326"))
    tiles: ee.FeatureCollection = ee.FeatureCollection(tile) #ADJUSTED TO SELECT SINGLE TILE

    # Get number of tiles
    num_tiles: int = tiles.size().getInfo() # tile_list #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES
    if num_tiles == 0:
        print("GTSM collection empty!")
        return

    # Get scale (if not specified)
    if scale == None:
        scale: float = tiler.zoom_to_scale(zoom).getInfo() # not specified, defaults to pre-set float
    
    # Get tasks
    task_list: List[ee.batch.Task] = []
    for date in dates.getInfo():
        if "subtidal" in mode:
            print('Subtidal mode not available')
        elif "intertidal" in mode:
            # Get subtidal bathymetry for tiles
            sdb_tiles: ee.ImageCollection = tiles.map(
                lambda tile: get_tile_intertidal_bathymetry(
                    tile=tile,
                    start=ee.String(date["start"]),
                    stop=ee.String(date["stop"])
                )#.clip(geometry)#.select('ndwi').rename('water_score') # clip individual tiles to match geometry of aoi, select ndwi and rename
            )

    # Convert tiles to list
    tile_list: ee.List = tiles.toList(num_tiles)

    # Export tiles
    task_list = export_sdb_tiles(
        sink=sink,
        tile_list=tile_list, # tile_list_up
        num_tiles=num_tiles,
        mode=mode,
        export_scale=scale,
        crs=crs,
        sdb_tiles=sdb_tiles, # sdb_tiles_up
        name_suffix=f"t{date['start']}_{date['stop']}_{scale}m",
        task_list=task_list,
        overwrite=overwrite,
        bucket=bucket
    )

    return task_list # toggle off when you need more dates to be run..

In [292]:
# Compute intertidal bathymetry for each tile. When tasks are submitted, check progress at:
# https://code.earthengine.google.com/tasks or https://console.cloud.google.com/earth-engine/tasks?project=bathymetry

tasks = []
for idx, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
    # Get tile
    ee_tile = ee.Geometry(row['geometry'].__geo_interface__, gdf_tiles.crs.to_string(), False)

    # Get properties
    ee_properties = {'tx': ee.String(str(row['tx'])), 'ty': ee.String(str(row['ty'])), 'zoom': ee.String(str(row['zoom'])),
                     'nearest_station_id': ee.String(row['nearest_station_id']), 'nearest_station_distance': ee.Number(row['nearest_station_distance']),
                     'nearest_station_latitude': ee.Number(row['nearest_station_latitude']), 'nearest_station_longitude': ee.Number(row['nearest_station_longitude'])}
    
    # Create feature
    ee_feature = ee.Feature(ee_tile).set(ee_properties)

    # Export tiles
    task = export_tiles(sink='cloud', mode=mode, geometry=ee_feature, zoom=zoom_level, start=start_date, stop=stop_date,
                        scale=scale, crs=crs, buf_pix=5, step_months=compo_int, window_months=compo_len, overwrite=True, bucket=bucket)
    
    # Append taks
    tasks.append(task)

# Get start time
start_time = time.time()

# save the task list as a pickle file
with open(os.path.join(file_path_progress, ("tasks_" + project_name +"2.pkl")), "wb") as f:
    pickle.dump(tasks, f)

  0%|          | 0/118 [00:00<?, ?it/s]

Submitting task for tile:  intertidal_improved_100m_global/z10/x199/y240/t2021-01-01_2022-01-01_100m


  1%|          | 1/118 [00:04<08:11,  4.20s/it]

Submitting task for tile:  intertidal_improved_100m_global/z10/x110/y297/t2021-01-01_2022-01-01_100m


  2%|▏         | 2/118 [00:07<06:36,  3.42s/it]

...

 98%|█████████▊| 116/118 [05:55<00:07,  3.66s/it]

Submitting task for tile:  intertidal_improved_100m_global/z10/x60/y318/t2021-01-01_2022-01-01_100m


 99%|█████████▉| 117/118 [05:58<00:03,  3.52s/it]

Submitting task for tile:  intertidal_improved_100m_global/z10/x127/y315/t2021-01-01_2022-01-01_100m


100%|██████████| 118/118 [06:02<00:00,  3.07s/it]


In [195]:
len(tasks)

61

In [196]:
# save the task list as a pickle file
with open(os.path.join(file_path_progress, ("tasks_" + project_name +"2.pkl")), "wb") as f:
    pickle.dump(tasks, f)

In [278]:
# Monitor tasks
project_name = "NWN"

# open the task list as a pickle file
with open(os.path.join(file_path_progress, ("tasks_" + project_name +"1.pkl")), "rb") as f:
    tasks = pickle.load(f)

n_tasks_failed, n_tasks_complete, n_tasks = 0, 0, 1
while n_tasks_failed + n_tasks_complete < n_tasks:
    # Get number of tasks
    n_tasks = len([task for tasks_ in tasks for task in tasks_])
    
    # Get task statuses
    task_statuses = [task.status() for tasks_ in tasks for task in tasks_]

    # Get number of tasks running, completed and failed
    n_tasks_ready = sum([task_status['state'] == 'READY' for task_status in task_statuses])
    n_tasks_running = sum([task_status['state'] == 'RUNNING' for task_status in task_statuses])
    n_tasks_complete = sum([task_status['state'] == 'COMPLETED' for task_status in task_statuses])
    n_tasks_failed = sum([task_status['state'] == 'FAILED' for task_status in task_statuses])

    # Get time elapsed
    time_elapsed = time.time() - start_time

    # Print tasks
    print('Tasks: {} ready, {} running, {} complete, {} failed (after {:.2f} minutes)'.format(n_tasks_ready, n_tasks_running, n_tasks_complete, n_tasks_failed, time_elapsed / 60), end='\r')

    # Wait for 10 seconds
    time.sleep(10)

# Print tasks
print('Tasks: ready {}, {} running, {} complete, {} failed (after {:.2f} minutes)'.format(n_tasks_ready, n_tasks_running, n_tasks_complete, n_tasks_failed, time_elapsed / 60))

KeyboardInterrupt: 